# Time Series Feature Selection: Avoiding Data Leakage

## What You Will Do
You will build a synthetic financial time series, deliberately leak future information using standard cross-validation, observe the inflated accuracy, then fix it with walk-forward feature selection using a purged time series split. The difference in reported vs. real performance is the leakage tax you pay when you ignore time structure.

## The Core Problem
Standard k-fold cross-validation shuffles data randomly. In a time series, shuffling means your training fold can contain data from the future relative to your test fold. A feature selected or trained on shuffled folds appears to have predictive power it does not actually possess.

## Estimated Time: under 2 minutes

---

## Setup

In [ ]:
# Purpose: Import all dependencies and seed RNG
# Key Concept: TimeSeriesSplit respects temporal order; KFold does not

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Setup complete.")

## Build a Synthetic Financial Time Series

We create a dataset that mimics a real trading signal problem:
- **Target**: next-day return direction (up = 1, down = 0) for a simulated asset
- **Features**: rolling momentum, volatility, mean reversion, and volume proxies computed from synthetic price data
- **Spurious features**: lagged realisations of the future return (these should be invisible to a correctly designed selector but become visible when we shuffle)

The spurious features are what make leakage dangerous — a shuffled CV will rank them as highly informative.

In [ ]:
# Purpose: Generate a synthetic financial time series with known causal and spurious features
# Key Concept: We engineer the data so we know ground truth — which features are
#              genuinely predictive and which only appear predictive under leakage

def make_financial_ts(n_obs=800, random_state=42):
    """
    Create a synthetic financial classification dataset with temporal structure.

    Returns a DataFrame with features and a binary target column.
    Spurious features are future leaks embedded in feature space.
    """
    rng = np.random.RandomState(random_state)

    # Simulate log-returns with a slight momentum regime
    returns = rng.randn(n_obs) * 0.01
    # Regime: slow-moving trend component
    trend = np.cumsum(rng.randn(n_obs) * 0.002)
    returns = returns + 0.3 * np.gradient(trend)

    price = 100 * np.exp(np.cumsum(returns))
    dates = pd.date_range('2018-01-01', periods=n_obs, freq='B')

    df = pd.DataFrame({'price': price, 'returns': returns}, index=dates)

    # --- Causal features: computed strictly from past data ---
    # 5-day momentum
    df['momentum_5'] = df['returns'].rolling(5).mean()
    # 20-day volatility
    df['volatility_20'] = df['returns'].rolling(20).std()
    # Normalised distance from 20-day moving average
    ma20 = df['price'].rolling(20).mean()
    df['mean_reversion'] = (df['price'] - ma20) / ma20
    # Simulated volume proxy (autocorrelated)
    volume = 1e6 + rng.randn(n_obs).cumsum() * 1e4
    df['volume_ratio'] = pd.Series(volume, index=dates).rolling(5).mean() / volume

    # --- Spurious features: they contain future information ---
    # future_ret_1 = next day's return (pure future leak)
    df['future_ret_1'] = df['returns'].shift(-1)
    # future_ret_2 = average of next 3 days (another leak)
    df['future_ret_3'] = df['returns'].shift(-1).rolling(3).mean().shift(-2)

    # Target: is next-day return positive?
    df['target'] = (df['returns'].shift(-1) > 0).astype(int)

    # Drop NaN rows from rolling windows
    df = df.dropna()

    feature_cols = ['momentum_5', 'volatility_20', 'mean_reversion',
                    'volume_ratio', 'future_ret_1', 'future_ret_3']
    return df[feature_cols], df['target'], feature_cols


X, y, feature_names = make_financial_ts(n_obs=800)
print(f"Dataset shape: {X.shape}")
print(f"Date range  : {X.index[0].date()} to {X.index[-1].date()}")
print(f"Features    : {feature_names}")
print(f"Target balance: {y.mean():.2%} positive days")

## Step 1 — The Leakage Problem: Standard CV

Here we use `KFold` (the default scikit-learn CV) which shuffles data before splitting. We then run mutual information ranking on one fold and evaluate on another — but those folds can contain overlapping time periods in reverse order.

Watch how the spurious future-leak features score **higher** than the causal features.

In [ ]:
# Purpose: Demonstrate feature selection with standard KFold — the leaky baseline
# Key Concept: KFold ignores time order; future information bleeds into training folds

Xn = X.values
yn = y.values

kfold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Collect MI scores across all folds (simulate selection-before-training pipeline)
mi_scores_kfold = np.zeros(len(feature_names))
for train_idx, _ in kfold.split(Xn):
    fold_mi = mutual_info_classif(Xn[train_idx], yn[train_idx], random_state=RANDOM_STATE)
    mi_scores_kfold += fold_mi
mi_scores_kfold /= 5

mi_ranking_kfold = np.argsort(mi_scores_kfold)[::-1]

print("Feature ranking with STANDARD (shuffled) KFold:")
print(f"  {'Rank':<6} {'Feature':<20} {'MI Score':>10}  {'Type'}")
print('  ' + '-' * 55)
for rank, idx in enumerate(mi_ranking_kfold, start=1):
    feat_type = 'LEAK (future)' if 'future' in feature_names[idx] else 'causal'
    print(f"  {rank:<6} {feature_names[idx]:<20} {mi_scores_kfold[idx]:>10.4f}  {feat_type}")

## Step 2 — The Fix: Walk-Forward CV with Purging

`TimeSeriesSplit` always trains on data **before** the test fold. This prevents future data from appearing in training. We add a **purge gap** of 5 days between train and test to handle datasets where overlapping rolling windows could still carry forward-looking information through the boundary.

With proper temporal splits, the MI scores of future-leak features collapse to near zero.

In [ ]:
# Purpose: Demonstrate walk-forward feature selection with a purge gap
# Key Concept: Purging removes boundary observations that could carry future
#              information through overlapping rolling windows

def purged_time_series_splits(n_samples, n_splits=5, purge_gap=5):
    """
    Yield (train_idx, test_idx) pairs where test is always strictly after train,
    with a purge_gap of observations excluded from training to prevent
    boundary leakage through rolling-window features.

    Parameters
    ----------
    n_samples : int
    n_splits : int
    purge_gap : int
        Number of observations to drop from the end of each training set
    """
    tscv = TimeSeriesSplit(n_splits=n_splits)
    for train_idx, test_idx in tscv.split(np.arange(n_samples)):
        purged_train = train_idx[:-purge_gap] if len(train_idx) > purge_gap else train_idx
        yield purged_train, test_idx


# Collect MI scores using walk-forward splits
mi_scores_wf = np.zeros(len(feature_names))
n_folds_used = 0
for train_idx, _ in purged_time_series_splits(len(Xn), n_splits=5, purge_gap=5):
    fold_mi = mutual_info_classif(Xn[train_idx], yn[train_idx], random_state=RANDOM_STATE)
    mi_scores_wf += fold_mi
    n_folds_used += 1
mi_scores_wf /= n_folds_used

mi_ranking_wf = np.argsort(mi_scores_wf)[::-1]

print("Feature ranking with WALK-FORWARD (purged) CV:")
print(f"  {'Rank':<6} {'Feature':<20} {'MI Score':>10}  {'Type'}")
print('  ' + '-' * 55)
for rank, idx in enumerate(mi_ranking_wf, start=1):
    feat_type = 'LEAK (future)' if 'future' in feature_names[idx] else 'causal'
    print(f"  {rank:<6} {feature_names[idx]:<20} {mi_scores_wf[idx]:>10.4f}  {feat_type}")

## Step 3 — How Selected Features Change Over Time Windows

Markets are non-stationary. A feature that was informative in 2018 may not be in 2020. Walk-forward selection lets us track which features matter in each time window — static selection misses this entirely.

In [ ]:
# Purpose: Show how feature importance evolves across time windows
# Key Concept: Non-stationarity means the optimal feature set shifts over time;
#              static selection is a snapshot, not a truth

# Split the dataset into 5 roughly equal time windows and rank features in each
n_windows = 5
window_size = len(Xn) // n_windows

window_rankings = []
window_labels = []
causal_features = [f for f in feature_names if 'future' not in f]
causal_idx = [feature_names.index(f) for f in causal_features]

for w in range(n_windows):
    start = w * window_size
    end = start + window_size
    X_win = Xn[start:end, :][:, causal_idx]  # only causal features
    y_win = yn[start:end]

    mi_win = mutual_info_classif(X_win, y_win, random_state=RANDOM_STATE)
    window_rankings.append(mi_win)

    period_start = X.index[start].strftime('%b %Y')
    period_end = X.index[min(end - 1, len(X) - 1)].strftime('%b %Y')
    window_labels.append(f"W{w+1}\n{period_start}")

rankings_matrix = np.array(window_rankings)  # shape: (n_windows, n_causal_features)

# Visualise as a heatmap
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(rankings_matrix.T, aspect='auto', cmap='YlOrRd')
ax.set_yticks(range(len(causal_features)))
ax.set_yticklabels(causal_features, fontsize=10)
ax.set_xticks(range(n_windows))
ax.set_xticklabels(window_labels, fontsize=9)
ax.set_xlabel('Time Window', fontsize=11)
ax.set_title('Mutual Information by Feature Across Time Windows\n'
             '(brighter = more informative in that window)', fontsize=12)
plt.colorbar(im, ax=ax, label='MI Score')
plt.tight_layout()
plt.show()

print("Observation: feature importance is not constant over time.")
print("Static selection picks the average; walk-forward adapts.")

## Step 4 — Performance Comparison: Leaky vs Walk-Forward

We train a Gradient Boosting model under both CV regimes. The leaky pipeline includes the future-leak features; the walk-forward pipeline selects only causal features using purged splits. The accuracy gap is the leakage inflation.

In [ ]:
# Purpose: Quantify the performance difference between leaky and clean evaluation
# Key Concept: Leaky evaluation produces optimistic estimates that do not transfer
#              to live trading; the gap is the cost you pay when you deploy

gb = GradientBoostingClassifier(n_estimators=50, max_depth=3, random_state=RANDOM_STATE)
scaler = StandardScaler()

# Leaky: all features, shuffled KFold
kfold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores_leaky = cross_val_score(gb, scaler.fit_transform(Xn), yn, cv=kfold, scoring='accuracy')

# Walk-forward: causal features only, purged TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)
Xn_causal = Xn[:, causal_idx]
Xs_causal = scaler.fit_transform(Xn_causal)

wf_scores = []
for train_idx, test_idx in purged_time_series_splits(len(Xn), n_splits=5, purge_gap=5):
    gb.fit(Xs_causal[train_idx], yn[train_idx])
    wf_scores.append((gb.predict(Xs_causal[test_idx]) == yn[test_idx]).mean())
wf_scores = np.array(wf_scores)

print(f"{'Evaluation regime':<40} {'Mean Acc':>9} {'Std':>7} {'Features':>10}")
print('-' * 70)
print(f"{'Leaky (KFold + future features)':<40} {scores_leaky.mean():>9.4f} "
      f"{scores_leaky.std():>7.4f} {Xn.shape[1]:>10}")
print(f"{'Walk-forward (purged, causal only)':<40} {wf_scores.mean():>9.4f} "
      f"{wf_scores.std():>7.4f} {len(causal_idx):>10}")
print()
inflation = scores_leaky.mean() - wf_scores.mean()
print(f"Leakage inflation: {inflation:+.4f} ({inflation*100:.2f} percentage points)")
print("This inflation would lead to a strategy that looks profitable in backtest but fails live.")

## Visualisation — Leakage in Action

In [ ]:
# Purpose: Side-by-side chart of leaky vs honest feature ranks and performance
# Key Concept: A picture makes the divergence impossible to dismiss

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: MI scores comparison for causal features
ax = axes[0]
x = np.arange(len(causal_features))
kfold_causal = mi_scores_kfold[causal_idx]
wf_causal = mi_scores_wf[causal_idx]

width = 0.35
ax.bar(x - width/2, kfold_causal, width, label='KFold (leaky)', color='#d62728', alpha=0.85)
ax.bar(x + width/2, wf_causal, width, label='Walk-forward (honest)', color='#2ca02c', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(causal_features, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Mutual Information Score', fontsize=11)
ax.set_title('MI Scores: Leaky vs Walk-Forward\n(leak features excluded from this chart)', fontsize=11)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

# Right: Accuracy box plots
ax2 = axes[1]
bp = ax2.boxplot(
    [scores_leaky, wf_scores],
    labels=['Leaky\n(KFold + future features)', 'Honest\n(walk-forward, causal)'],
    patch_artist=True,
    boxprops=dict(facecolor='#aec7e8'),
    medianprops=dict(color='red', linewidth=2)
)
bp['boxes'][0].set_facecolor('#d62728')
bp['boxes'][0].set_alpha(0.6)
bp['boxes'][1].set_facecolor('#2ca02c')
bp['boxes'][1].set_alpha(0.6)
ax2.set_ylabel('Fold Accuracy', fontsize=11)
ax2.set_title('Walk-Forward Accuracy is Lower but Honest\n'
              f'(leakage inflation = {inflation*100:.1f} pp)', fontsize=11)
ax2.axhline(y=0.5, color='grey', linestyle='--', alpha=0.5, label='Random baseline')
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Exercise: Tune the Purge Gap

**Task:** The purge gap of 5 days was chosen because our rolling windows span up to 20 days. Investigate what happens to walk-forward accuracy and feature rankings as you increase the gap from 0 to 25.

**Expected outcome:** You should observe that gap=0 slightly inflates accuracy for features with long rolling windows, and that beyond ~20 days the effect plateaus.

<details>
<summary>Hint 1</summary>
Loop over gap values [0, 5, 10, 15, 20, 25] and call purged_time_series_splits with each. Plot mean walk-forward accuracy vs gap size.
</details>

<details>
<summary>Hint 2 (more specific)</summary>
The purge gap should be at least as large as the longest rolling window used to construct features. For a 20-day volatility feature, set gap >= 20.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
# Experiment with purge gap sizes

gap_values = [0, 5, 10, 15, 20, 25]  # Modify this list
gap_accuracies = []

for gap in gap_values:
    fold_scores = []
    for train_idx, test_idx in purged_time_series_splits(len(Xn), n_splits=5, purge_gap=gap):
        if len(train_idx) < 10:
            continue
        gb.fit(Xs_causal[train_idx], yn[train_idx])
        fold_scores.append((gb.predict(Xs_causal[test_idx]) == yn[test_idx]).mean())
    gap_accuracies.append(np.mean(fold_scores))
    print(f"Gap = {gap:3d}: mean accuracy = {gap_accuracies[-1]:.4f}")

optimal_gap = gap_values[np.argmax(gap_accuracies)]  # Note: max here is not the goal
your_chosen_gap = None  # Replace with your answer: what gap do you think is correct?
print(f"\nYour chosen purge gap: {your_chosen_gap}")

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise():
    assert len(gap_accuracies) == len(gap_values), "Run the loop above for all gap values."
    assert your_chosen_gap is not None, "Set your_chosen_gap to a gap value (integer)."
    assert isinstance(your_chosen_gap, int), "your_chosen_gap should be an integer."
    assert your_chosen_gap >= 0, "Gap cannot be negative."
    # The correct answer is: gap should be >= the longest rolling window (20 days)
    assert your_chosen_gap >= 10, (
        f"Gap {your_chosen_gap} is too small. The longest feature window is 20 days; "
        "purge gap should be at least 10-20 to prevent boundary leakage."
    )
    print(f"Exercise passed! Chosen purge gap: {your_chosen_gap} days — a sound choice.")

test_exercise()

## Summary

**Key Takeaways:**

1. **Standard KFold shuffles time** — future data leaks into training folds, inflating all accuracy estimates.
2. **Leaky feature selection promotes future-leak features** to the top of rankings, where they appear highly informative.
3. **Walk-forward splits** respect temporal order: training always precedes testing.
4. **Purging** removes boundary observations to prevent rolling-window features from carrying forward-looking signal across the fold boundary.
5. **Feature importance is non-stationary** — the optimal feature set changes over time windows.

**What is next:**
- `boruta_vs_shap_shootout.ipynb` — compare wrapper and interpretability-based methods
- Module 7 (Time Series Feature Selection) for purged CV with embargo periods and combinatorial splits